In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    VotingClassifier,
)
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.preprocessing import LabelEncoder, RobustScaler
import xgboost as xgb

warnings.filterwarnings("ignore")

print("=" * 70)
print("Bearing Fault Detection - Final Optimized Version")
print("=" * 70)

# ============================================================
# Phase 1: Data Loading and Basic Cleaning (Keep Outliers)
# ============================================================
print("\n" + "=" * 60)
print("Phase 1: Data Loading and Basic Cleaning")
print("=" * 60)

df = pd.read_csv("DataSetbearing-failure.csv")
print(f"Raw data shape: {df.shape}")

# Only remove NaN values - DO NOT remove outliers
df = df.dropna().reset_index(drop=True)
print(f"After removing NaN: {df.shape}")

# ============================================================
# Phase 2: Feature Engineering (Expanded but Controlled)
# ============================================================
print("\n" + "=" * 60)
print("Phase 2: Feature Engineering")
print("=" * 60)

eps = 1e-9
base_cols = [
    "Vel, Rms (RMS)",
    "Acc, Rms (RMS)",
    "Crest (RMS)",
    "Kurt (RMS)",
    "Vel, Peak (RMS)",
    "Vel, Peak to peak (RMS)",
]

# Basic ratios
df["peak_to_rms"] = df["Vel, Peak (RMS)"] / (df["Vel, Rms (RMS)"] + eps)
df["pp_to_rms"] = df["Vel, Peak to peak (RMS)"] / (df["Vel, Rms (RMS)"] + eps)
df["pp_to_peak"] = df["Vel, Peak to peak (RMS)"] / (df["Vel, Peak (RMS)"] + eps)
df["acc_vel_ratio"] = df["Acc, Rms (RMS)"] / (df["Vel, Rms (RMS)"] + eps)

# Fault indicators
df["crest_factor"] = df["Vel, Peak (RMS)"] / (df["Vel, Rms (RMS)"] + eps)
df["impulse_factor"] = df["Vel, Peak (RMS)"] / (np.abs(df["Vel, Rms (RMS)"]) + eps)
df["margin_factor"] = df["Vel, Peak (RMS)"] / ((np.square(np.sqrt(np.abs(df["Vel, Rms (RMS)"])))) + eps)
df["kurtosis_index"] = df["Kurt (RMS)"] / 3.0
df["severity_index"] = df["Vel, Rms (RMS)"] * df["Acc, Rms (RMS)"] * df["Crest (RMS)"]
df["early_fault_index"] = (df["Kurt (RMS)"] * df["Crest (RMS)"]) / (df["Acc, Rms (RMS)"] + eps)
df["composite_fault"] = (
    (df["Kurt (RMS)"] / 3.0) *
    (df["Crest (RMS)"] / (df["Crest (RMS)"].mean() + eps)) *
    (df["Vel, Peak (RMS)"] / (df["Vel, Rms (RMS)"] + eps))
)

# Transformations
for col in base_cols:
    df[f"{col}_log"] = np.log1p(np.maximum(0, df[col]))
    df[f"{col}_sq"] = df[col] ** 2

# Interactions
df["kurt_crest"] = df["Kurt (RMS)"] * df["Crest (RMS)"]
df["kurt_vel"] = df["Kurt (RMS)"] * df["Vel, Rms (RMS)"]
df["vel_acc"] = df["Vel, Rms (RMS)"] * df["Acc, Rms (RMS)"]

# Encode categorical variables
le_comp = LabelEncoder()
df["COMP_NAME_encoded"] = le_comp.fit_transform(df["COMP_NAME"].astype(str))

le_loc = LabelEncoder()
df["MP_LOC_encoded"] = le_loc.fit_transform(df["MP_LOC"].astype(str))

print(f"Total features created: {len(df.columns) - 3}")

# ============================================================
# Phase 3: Train/Val/Test Split with Group Statistics (No Leakage)
# ============================================================
print("\n" + "=" * 60)
print("Phase 3: Train/Val/Test Split with Group Statistics")
print("=" * 60)

X = df.drop(columns=["Label", "COMP_NAME", "MP_LOC"]).copy()
y = df["Label"].copy()

# Split before any group statistics
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# Compute group statistics ONLY from train data
X_train_grp = X_train.copy()
X_train_grp["COMP_NAME"] = df.loc[X_train.index, "COMP_NAME"].values
X_train_grp["MP_LOC"] = df.loc[X_train.index, "MP_LOC"].values

for col in base_cols:
    if col in X_train.columns:
        grp_means = X_train_grp.groupby(["COMP_NAME", "MP_LOC"])[col].mean()
        grp_stds = X_train_grp.groupby(["COMP_NAME", "MP_LOC"])[col].std() + eps
        
        overall_mean = X_train_grp[col].mean()
        overall_std = X_train_grp[col].std() + eps
        
        def add_group_stats(target_X, grp_means, grp_stds, overall_mean, overall_std, col):
            target_comp = df.loc[target_X.index, "COMP_NAME"].values
            target_loc = df.loc[target_X.index, "MP_LOC"].values
            keys = list(zip(target_comp, target_loc))
            
            m = np.array([grp_means.get(k, overall_mean) for k in keys])
            s = np.array([grp_stds.get(k, overall_std) for k in keys])
            
            target_X[f"{col}_grp_zscore"] = (target_X[col] - m) / (s + eps)
            return target_X
        
        X_train = add_group_stats(X_train, grp_means, grp_stds, overall_mean, overall_std, col)
        X_val = add_group_stats(X_val, grp_means, grp_stds, overall_mean, overall_std, col)
        X_test = add_group_stats(X_test, grp_means, grp_stds, overall_mean, overall_std, col)

print(f"Final shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

# ============================================================
# Phase 4: Feature Selection and Scaling
# ============================================================
print("\n" + "=" * 60)
print("Phase 4: Feature Selection and Scaling")
print("=" * 60)

n_features = min(35, X_train.shape[1])

try:
    rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
    rfe = RFE(estimator=rf_selector, n_features_to_select=n_features)
    
    X_train_selected = rfe.fit_transform(X_train, y_train)
    X_val_selected = rfe.transform(X_val)
    X_test_selected = rfe.transform(X_test)
    
    selected_features = X_train.columns[rfe.get_support()].tolist()
    print(f"RFE selected {len(selected_features)} features")
except Exception as e:
    print(f"RFE failed, using SelectKBest: {e}")
    selector = SelectKBest(f_classif, k=n_features)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_val_selected = selector.transform(X_val)
    X_test_selected = selector.transform(X_test)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_val_scaled = scaler.transform(X_val_selected)
X_test_scaled = scaler.transform(X_test_selected)

# ============================================================
# Phase 5: Hyperparameter Tuning
# ============================================================
print("\n" + "=" * 60)
print("Phase 5: Hyperparameter Tuning")
print("=" * 60)

param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.07],
    "n_estimators": [200, 300, 500],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
}

try:
    xgb_base = xgb.XGBClassifier(random_state=42, eval_metric="logloss", verbosity=0)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        xgb_base, param_grid, cv=cv,
        scoring="f1_macro", n_jobs=-1, verbose=0
    )
    grid_search.fit(X_train_scaled, y_train)
    best_params = grid_search.best_params_
    print(f"Best parameters: {best_params}")
    print(f"Best score: {grid_search.best_score_:.4f}")
except Exception as e:
    print(f"GridSearch failed: {e}")
    best_params = {
        "max_depth": 6,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    }
    print(f"Using default parameters: {best_params}")

# ============================================================
# Phase 6: Ensemble Model Training
# ============================================================
print("\n" + "=" * 60)
print("Phase 6: Ensemble Model Training")
print("=" * 60)

class OptimizedEnsemble:
    
    def __init__(self, best_params):
        self.best_params = best_params
        self.models = {}
        self.ensemble = None
        self.thresholds = {0: 1.0, 1: 1.0, 2: 1.0}
    
    def fit(self, X, y, X_val, y_val):
        # 1. XGBoost Primary
        print("  Training XGBoost 1...", end=" ")
        xgb1 = xgb.XGBClassifier(
            **self.best_params,
            random_state=42,
            eval_metric="logloss",
            verbosity=0
        )
        xgb1.fit(X, y)
        self.models["xgb1"] = xgb1
        print("OK")
        
        # 2. XGBoost with class weighting
        print("  Training XGBoost 2 (weighted)...", end=" ")
        xgb2 = xgb.XGBClassifier(
            **self.best_params,
            scale_pos_weight=1.3,
            random_state=123,
            eval_metric="logloss",
            verbosity=0
        )
        xgb2.fit(X, y)
        self.models["xgb2"] = xgb2
        print("OK")
        
        # 3. LightGBM
        print("  Training LightGBM...", end=" ")
        lgbm = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=8,
            class_weight="balanced",
            random_state=42,
            verbose=-1
        )
        lgbm.fit(X, y)
        self.models["lgbm"] = lgbm
        print("OK")
        
        # 4. RandomForest
        print("  Training RandomForest...", end=" ")
        rf = RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            class_weight="balanced_subsample",
            random_state=42
        )
        rf.fit(X, y)
        self.models["rf"] = rf
        print("OK")
        
        # 5. XGBoost Secondary
        print("  Training XGBoost 3 (secondary)...", end=" ")
        xgb3 = xgb.XGBClassifier(
            max_depth=8,
            learning_rate=0.07,
            n_estimators=200,
            subsample=0.7,
            colsample_bytree=0.7,
            random_state=456,
            eval_metric="logloss",
            verbosity=0
        )
        xgb3.fit(X, y)
        self.models["xgb3"] = xgb3
        print("OK")
        
        # Build ensemble
        print("  Building ensemble...", end=" ")
        estimators = [
            ("xgb1", xgb1),
            ("xgb2", xgb2),
            ("lgbm", lgbm),
            ("rf", rf),
            ("xgb3", xgb3),
        ]
        self.ensemble = VotingClassifier(
            estimators=estimators,
            voting="soft",
            weights=[1.2, 1.1, 1.0, 1.0, 1.1]
        )
        self.ensemble.fit(X, y)
        print("OK")
        
        # Optimize thresholds using validation set
        print("  Optimizing thresholds...", end=" ")
        probs_val = self.ensemble.predict_proba(X_val)
        best_macro_f1 = 0
        best_th = {0: 1.0, 1: 1.0, 2: 1.0}
        
        for w0 in np.arange(0.8, 1.3, 0.1):
            for w1 in np.arange(0.8, 1.3, 0.1):
                for w2 in np.arange(0.8, 1.3, 0.1):
                    weights_arr = np.array([w0, w1, w2])
                    adj_probs = probs_val * weights_arr
                    preds = np.argmax(adj_probs, axis=1)
                    
                    macro_f1 = f1_score(y_val, preds, average="macro")
                    if macro_f1 > best_macro_f1:
                        best_macro_f1 = macro_f1
                        best_th = {0: w0, 1: w1, 2: w2}
        
        self.thresholds = best_th
        print("OK")
        print(f"  Optimal thresholds: {self.thresholds}")
        print(f"  Best Macro F1 on validation: {best_macro_f1:.4f}")
        
        return self
    
    def predict(self, X):
        probs = self.ensemble.predict_proba(X)
        weights_arr = np.array([self.thresholds[0], self.thresholds[1], self.thresholds[2]])
        adj_probs = probs * weights_arr
        return np.argmax(adj_probs, axis=1)

# Train final model
model = OptimizedEnsemble(best_params)
model.fit(X_train_scaled, y_train, X_val_scaled, np.array(y_val))

# ============================================================
# Phase 7: Final Evaluation
# ============================================================
print("\n" + "=" * 60)
print("Phase 7: Final Evaluation on Test Data")
print("=" * 60)

y_pred = model.predict(X_test_scaled)

target_names = ["Class 0 (Healthy)", "Class 1 (Severe Fault)", "Class 2 (Mild Fault)"]

# Calculate all metrics
accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

precision_per_class = precision_score(y_test, y_pred, average=None)
recall_per_class = recall_score(y_test, y_pred, average=None)
f1_per_class = f1_score(y_test, y_pred, average=None)

precision_macro = precision_score(y_test, y_pred, average='macro')
precision_weighted = precision_score(y_test, y_pred, average='weighted')
recall_macro = recall_score(y_test, y_pred, average='macro')
recall_weighted = recall_score(y_test, y_pred, average='weighted')
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

mcc = matthews_corrcoef(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

print("\nConfusion Matrix:")
print("                 Predicted")
print("              Class0  Class1  Class2")
for i in range(3):
    print(f"Actual Class{i}   {cm[i,0]:6d}  {cm[i,1]:6d}  {cm[i,2]:6d}")

print("\n" + "=" * 60)
print("Summary Metrics:")
print("=" * 60)

metrics_df = pd.DataFrame({
    'Class': target_names,
    'Precision': [f"{p:.4f}" for p in precision_per_class],
    'Recall': [f"{r:.4f}" for r in recall_per_class],
    'F1-Score': [f"{f:.4f}" for f in f1_per_class],
    'Support': [np.sum(y_test == i) for i in range(3)]
})

print("\nPer-class metrics:")
print(metrics_df.to_string(index=False))

print("\nOverall metrics:")
print(f"  {'Metric':<20} {'Macro':<15} {'Weighted':<15}")
print("-" * 50)
print(f"  {'Precision':<20} {precision_macro:<15.4f} {precision_weighted:<15.4f}")
print(f"  {'Recall':<20} {recall_macro:<15.4f} {recall_weighted:<15.4f}")
print(f"  {'F1-Score':<20} {f1_macro:<15.4f} {f1_weighted:<15.4f}")

print("\nAdvanced metrics:")
print(f"  {'Accuracy':<20} {accuracy:<15.4f}")
print(f"  {'Balanced Accuracy':<20} {balanced_acc:<15.4f}")
print(f"  {'MCC':<20} {mcc:<15.4f}")
print(f"  {'Cohen Kappa':<20} {kappa:<15.4f}")

# Save results
with open("model_results_optimized.txt", "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("Bearing Fault Detection - Optimized Results\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Best parameters: {best_params}\n")
    f.write(f"Optimal thresholds: {model.thresholds}\n\n")
    
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Balanced Accuracy: {balanced_acc:.4f}\n")
    f.write(f"MCC: {mcc:.4f}\n")
    f.write(f"Cohen Kappa: {kappa:.4f}\n\n")
    
    f.write(metrics_df.to_string(index=False) + "\n\n")
    
    f.write(f"Precision (Macro): {precision_macro:.4f}\n")
    f.write(f"Precision (Weighted): {precision_weighted:.4f}\n")
    f.write(f"Recall (Macro): {recall_macro:.4f}\n")
    f.write(f"Recall (Weighted): {recall_weighted:.4f}\n")
    f.write(f"F1-Score (Macro): {f1_macro:.4f}\n")
    f.write(f"F1-Score (Weighted): {f1_weighted:.4f}\n\n")
    
    f.write(classification_report(y_test, y_pred, target_names=target_names))
    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))

print("\nResults saved to: model_results_optimized.txt")
print("\n" + "=" * 60)
print("Project completed successfully!")
print("=" * 60)
